## Correlation analysis

In [ ]:
# Prepare training data
features = data_train.drop(columns=['etiqueta', 'n_imagen', 'etiqueta_multi'])
target = data_train['etiqueta']

# Calculate the correlation matrix
correlation_matrix = features.corr()

# Create a heatmap
def plot_correlation_heatmap(matrix):
    # Apply font settings in Matplotlib
    plt.rcParams["font.family"] = 'serif'
    plt.figure(figsize=(16, 14))  # Adjust the figure size
    sns.heatmap(
        matrix,
        annot=False,  # Hide numerical annotations
        cmap="coolwarm",  # Color palette
        cbar=True,      # Show color bar
        # Color bar range from -1 to 1
        vmin=-1,
        vmax=1,
        center=0,
        xticklabels=True,  # Show labels on the x-axis
        yticklabels=True,  # Show labels on the y-axis
        square=True      # Keep squares proportional
    )
    plt.show()

# Usage
plot_correlation_heatmap(correlation_matrix)


In [ ]:
# Set the correlation threshold
threshold = 0.80

# Function to identify highly correlated variable pairs
def find_high_correlations(matrix, threshold):
    # Select pairs of variables with correlation above the threshold
    high_correlation = (matrix.where(matrix > threshold)
                        .stack()
                        .reset_index())

    # Filter duplicates (since correlation is symmetric) and remove self-correlations
    high_correlation = high_correlation[high_correlation['level_0'] != high_correlation['level_1']]

    # Rename columns for clarity
    high_correlation.columns = ['Variable1', 'Variable2', 'Correlation']

    # Remove inverse duplicates (mirror), e.g., (Variable1, Variable2) and (Variable2, Variable1)
    high_correlation = high_correlation.sort_values(by=['Variable1', 'Variable2']).drop_duplicates(subset=['Variable1', 'Variable2'])

    return high_correlation

# Usage
high_correlation = find_high_correlations(correlation_matrix, threshold)

# Display the result
for index, row in high_correlation.iterrows():
    print(f"Variables: {row['Variable1']} and {row['Variable2']}, Correlation: {row['Correlation']:.2f}")


In [ ]:
# Initialize a list to store columns to remove
columns_to_remove = []

# Iterate over the correlation matrix
for i in range(len(correlation_matrix.columns)):
    for j in range(i):
        if correlation_matrix.iloc[i, j] > threshold:  # If correlation is above the threshold
            colname = correlation_matrix.columns[i]   # Name of the highly correlated column
            columns_to_remove.append(colname)            # Add to the list for removal

# Remove duplicates from the list
columns_to_remove = list(set(columns_to_remove))
print(f"Number of columns to remove: {len(columns_to_remove)}")


In [ ]:
# Remove columns
data_train=data_train.drop(columns=columns_to_remove)
data_test=data_test.drop(columns=columns_to_remove)

In [ ]:
# Identify constant columns
constant_columns = []
for column in data_train.columns:
    if data_train[column].nunique() == 1:
        constant_columns.append(column)
print(constant_columns)


In [ ]:
# Remove constant columns
data_train=data_train.drop(columns=columnas_constantes)
data_test=data_test.drop(columns=columnas_constantes)

In [ ]:
data_train.columns

In [ ]:
data_train.shape